In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re

In [2]:
perla_final = pd.read_excel('/content/perla_final.xlsx')

In [3]:
copy_df = perla_final.copy()

In [4]:
def parse_basis(x):
    if pd.isna(x):
        return pd.NA

    values = [item.strip() for item in str(x).split("|")]

    values = [
        "Absorption" if item.lower() == "absorption" else item
        for item in values
    ]

    return " | ".join(values)


copy_df["Perovskite_band_gap_estimation_basis"] = copy_df["Perovskite_band_gap_estimation_basis"].apply(parse_basis)


"""
perovskite_parser.py
====================
Parsers for the composition columns of the Perovskite Database (v5.4 description):

    Perovskite_composition_short_form
    Perovskite_composition_long_form
    Perovskite_composition_a_ions / _a_ions_coefficients
    Perovskite_composition_b_ions / _b_ions_coefficients
    Perovskite_composition_c_ions / _c_ions_coefficients
    Perovskite_band_gap

Format rules taken from the database description
------------------------------------------------
* Layers are separated by ' | '        ->  "MA | Cs"      = 2 layers
* Ions / coefficients by ';'           ->  "Cs; FA; MA"   = 3 ions in one layer
* Coefficient lists must line up 1:1 with the ion lists (same layers, same order)
* Unknown coefficient is 'x'; '1-x' should have been written as 'x'
* A-/C-site labels of >= 3 characters should be wrapped in parentheses: (PEA)
* Short / long form are *derived* from the ion columns -> used here as cross-checks
* Band gap: one number per layer, layers separated by ' | '

Every parser returns the parsed value AND a set of flags that describe each abnormality
that was met, so nothing is silently "fixed".

Usage
-----
    from perovskite_parser import parse_perovskite_columns, flag_report
    parsed = parse_perovskite_columns(copy_df)
    print(flag_report(parsed))
"""
from __future__ import annotations

import math
import re
from functools import lru_cache

import numpy as np
import pandas as pd

NAN = float("nan")

COLS = dict(
    short="Perovskite_composition_short_form",
    long="Perovskite_composition_long_form",
    a="Perovskite_composition_a_ions",
    ac="Perovskite_composition_a_ions_coefficients",
    b="Perovskite_composition_b_ions",
    bc="Perovskite_composition_b_ions_coefficients",
    c="Perovskite_composition_c_ions",
    cc="Perovskite_composition_c_ions_coefficients",
    bg="Perovskite_band_gap",
)

MISSING_WORDS = {"", "nan", "none", "null", "n/a", "na"}
UNKNOWN_WORDS = {"unknown", "<unknown>"}
# anions that are legitimate on the C-site (labels are compared without parentheses)
VALID_C_IONS = {"I", "Br", "Cl", "F", "O", "S", "Se", "Te", "N", "SCN", "BF4", "PF6", "X"}

# ----------------------------------------------------------------------------------------
# low-level helpers
# ----------------------------------------------------------------------------------------


def _to_str(x):
    """None for every flavour of missing value, otherwise stripped string."""
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    s = str(x).strip()
    return None if s.lower() in MISSING_WORDS else s


def _split_top(s: str, seps: str):
    """Split on any char in `seps`, but never inside parentheses, e.g. '(1.3-Pr(NH3)2)'."""
    parts, buf, depth = [], [], 0
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth = max(depth - 1, 0)
        if depth == 0 and ch in seps:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    parts.append("".join(buf))
    return [p.strip() for p in parts]


def _outer_parens(t: str) -> bool:
    """True if the first '(' is closed by the very last character."""
    if not (t.startswith("(") and t.endswith(")")):
        return False
    depth = 0
    for i, ch in enumerate(t):
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
            if depth == 0 and i < len(t) - 1:
                return False
    return depth == 0


def _split_layers(s: str, flags: set):
    """'a; b | c' -> [['a','b'],['c']]  (handles ',' separators, trailing ';', empty layers)."""
    layers = []
    for lay in _split_top(s, "|"):
        toks = _split_top(lay, ";,")
        if len(toks) != len(_split_top(lay, ";")):
            flags.add("comma_instead_of_semicolon")
        while toks and toks[-1] == "":
            toks.pop()
            flags.add("trailing_separator")
        if not toks:
            flags.add("empty_layer")
        layers.append(toks)
    return layers


# ----------------------------------------------------------------------------------------
# ion columns  (a_ions, b_ions, c_ions)
# ----------------------------------------------------------------------------------------


def _clean_ion(tok: str, site: str, flags: set):
    if tok == "":
        flags.add("empty_token")
        return None
    low = tok.lower()
    if low in UNKNOWN_WORDS:
        flags.add("unknown_ion")
        return None
    if low in MISSING_WORDS:
        flags.add("nan_token")
        return None
    paren = _outer_parens(tok)
    name = tok[1:-1].strip() if paren else tok
    # rule from the description: labels with >= 3 characters go in parentheses (A and C site)
    if not paren and len(name) >= 3 and site in ("A", "C"):
        flags.add("long_label_without_parentheses")
    if site == "C" and name not in VALID_C_IONS:
        flags.add("unexpected_c_site_ion")        # e.g. 'Cs; FA; MA' typed into the C column
    if name == "X":
        flags.add("placeholder_ion")
    return name


@lru_cache(maxsize=None)
def _parse_ions_cached(s, site):
    if s is None:
        return (), frozenset({"missing"})
    flags = set()
    layers = [[_clean_ion(t, site, flags) for t in toks] for toks in _split_layers(s, flags)]
    if len(layers) > 1:
        flags.add("multi_layer")
    return tuple(tuple(l) for l in layers), frozenset(flags)


def parse_ions(value, site):
    """'Cs; FA | (PEA)'  ->  ([['Cs','FA'], ['PEA']], flags)"""
    layers, flags = _parse_ions_cached(_to_str(value), site)
    return [list(l) for l in layers], set(flags)


# ----------------------------------------------------------------------------------------
# coefficient columns  (a_ions_coefficients, b_..., c_...)
# ----------------------------------------------------------------------------------------

_NUM = r"(?:\d+\.?\d*|\.\d+)"
_RE_LEADZERO = re.compile(r"^0\d+(?:\.\d+)?$")                      # 083, 095, 016, 02, 01.51
_RE_PLAIN = re.compile(rf"^{_NUM}$")
_RE_RANGE = re.compile(rf"^({_NUM})\s*-\s*({_NUM})$")                # 0.05-0.10
_RE_INEQ = re.compile(rf"^[<>]=?\s*({_NUM})$")                       # <1
_RE_SYMBOLIC = re.compile(r"^[0-9xyznabcw+\-*/().\s]*[xyznabcw][0-9xyznabcw+\-*/().\s]*$", re.I)


def _parse_coef_token(tok: str):
    """-> (float value, kind).  kind != 'numeric' is reported as a flag."""
    t = tok.strip()
    low = t.lower()
    if low == "":
        return NAN, "empty_token"
    if low in UNKNOWN_WORDS:
        return NAN, "unknown_coefficient"
    if low in MISSING_WORDS:
        return NAN, "nan_token"
    if _RE_LEADZERO.match(t):                       # '083' -> 0.83 ; '01.51' -> 1.51
        return (float(t) if "." in t else float("0." + t[1:])), "leading_zero_typo"
    if _RE_PLAIN.match(t):
        return float(t.rstrip(".")), "numeric"
    m = _RE_RANGE.match(t)
    if m:                                           # the description says: give best estimate
        return round((float(m.group(1)) + float(m.group(2))) / 2, 6), "range_midpoint"
    if _RE_INEQ.match(t):
        return NAN, "inequality"
    if _RE_SYMBOLIC.match(t):                       # x, y, 1-x, 2x, 3(1-x), 3x-y, n-1 ...
        return NAN, "symbolic"
    return NAN, "text_value"                        # trace, variable, dominant, surface modification


def _repair_percent(vals, site, flags):
    """A/B-site coefficients like [0.05, 83, 17] are fractions typed as percent.
    Repaired ONLY if every token is either <1 or >=10 AND (small + big/100) ~ 1 (+-0.06).
    Layered 2D entries such as [2, 0.725, 23.925, 4.35] are therefore left alone."""
    if site not in ("A", "B") or len(vals) < 2:
        return vals
    if not all(np.isfinite(v) for v in vals):
        return vals
    big = [v for v in vals if v >= 10]
    small = [v for v in vals if 0 <= v < 1]
    if big and small and len(big) + len(small) == len(vals):
        if abs(sum(small) + sum(big) / 100 - 1) <= 0.06:
            flags.add("percent_typo_repaired")
            return [v / 100 if v >= 10 else v for v in vals]
    return vals


@lru_cache(maxsize=None)
def _parse_coefs_cached(s, site):
    if s is None:
        return (), frozenset({"missing"})
    flags, layers = set(), []
    for toks in _split_layers(s, flags):
        vals = []
        for t in toks:
            v, kind = _parse_coef_token(t)
            if kind != "numeric":
                flags.add(kind)
            vals.append(v)
        layers.append(_repair_percent(vals, site, flags))
    if len(layers) > 1:
        flags.add("multi_layer")
    return tuple(tuple(l) for l in layers), frozenset(flags)


def parse_coefficients(value, site):
    """'0.05; 0.79; 0.16 | 2' -> ([[0.05,0.79,0.16],[2.0]], flags). Symbolic -> NaN."""
    layers, flags = _parse_coefs_cached(_to_str(value), site)
    return [list(l) for l in layers], set(flags)


def check_site(ion_layers, coef_layers, site):
    """Row-level rules that need BOTH the ion list and the coefficient list."""
    flags = set()
    if not ion_layers and not coef_layers:
        return flags
    if ion_layers and not coef_layers:
        flags.add("coefficients_missing_for_ions")
        return flags
    if coef_layers and not ion_layers:
        flags.add("ions_missing_for_coefficients")
        return flags
    if len(ion_layers) != len(coef_layers):
        flags.add("layer_count_mismatch")
    for ions, coefs in zip(ion_layers, coef_layers):
        if len(ions) != len(coefs):
            flags.add("ion_coefficient_count_mismatch")
        expected = {"A": 1.0, "B": 1.0, "C": 3.0}[site]
        if coefs and all(np.isfinite(c) for c in coefs) and max(coefs) <= expected:
            if abs(sum(coefs) - expected) > 0.05:
                flags.add(f"sum_not_{int(expected)}")      # informational only
    return flags


# ----------------------------------------------------------------------------------------
# short form / long form
# ----------------------------------------------------------------------------------------


def build_ion_vocab(df, ion_cols):
    """All ion labels that occur in the A/B/C columns, longest first (used for tokenising)."""
    vocab = set()
    for col in ion_cols:
        for v in df[col].dropna().unique():
            for lay in _split_top(str(v), "|"):
                for tok in _split_top(lay, ";,"):
                    t = tok[1:-1].strip() if _outer_parens(tok) else tok
                    if t and t.lower() not in MISSING_WORDS | UNKNOWN_WORDS:
                        vocab.add(t)
    return sorted(vocab, key=lambda s: (-len(s), s))


_PAREN_GROUP = r"\((?:[^()]|\([^()]*\))*\)"
_COEF_RAW = r"(?:(?:\d+\.?\d*|\.\d+)?(?:-?[xyz])?)"


def _make_regexes(vocab):
    alt = "|".join(re.escape(v) for v in vocab)
    ion = rf"{_PAREN_GROUP}|nan|{alt}|[A-Z][a-z]?" if alt else rf"{_PAREN_GROUP}|nan|[A-Z][a-z]?"
    return (re.compile(rf"(?:{ion})"),
            re.compile(rf"(?P<ion>{ion})(?P<coef>{_COEF_RAW})"))


def _norm_formula(s: str) -> str:
    return re.sub(r"nan|[\s(),]", "", s)


def parse_short_form(value, tok_re):
    """'(PEA)MAPbI | CsPbI' -> ([['PEA','MA','Pb','I'],['Cs','Pb','I']], flags)"""
    s = _to_str(value)
    if s is None:
        return [], {"missing"}
    flags, layers = set(), []
    for lay in _split_top(s, "|"):
        if "," in lay:
            flags.add("comma_in_formula")
            lay = lay.replace(",", "")
        toks, pos = [], 0
        while pos < len(lay):
            m = tok_re.match(lay, pos)
            if not m or m.end() == pos:
                flags.add("short_form_unparsed_chars")
                pos += 1
                continue
            t = m.group(0)
            pos = m.end()
            if t == "nan":
                flags.add("nan_in_formula")      # an ion list was empty when the formula was built
                continue
            toks.append(t[1:-1] if _outer_parens(t) else t)
        layers.append(toks)
    if len(layers) > 1:
        flags.add("multi_layer")
    return layers, flags


def parse_long_form(value, long_re, vocab_set):
    """'Cs0.05FA0.76MA0.16PbBr1.5I1.5' -> [[('Cs',0.05),('FA',0.76),...]]
    Coefficient omitted -> 1.0.  Sub-groups such as Pb(I0.83Br0.17)3 are expanded.
    Symbolic coefficients (x, 1-x) -> NaN."""
    s = _to_str(value)
    if s is None:
        return [], {"missing"}
    flags, layers = set(), []
    for lay in _split_top(s, "|"):
        pairs, pos = [], 0
        while pos < len(lay):
            m = long_re.match(lay, pos)
            if not m or m.end() == pos:
                flags.add("long_form_unparsed_chars")
                pos += 1
                continue
            pos = m.end()
            ion, raw = m.group("ion"), m.group("coef")
            if ion == "nan":
                flags.add("nan_in_formula")
                continue
            val, kind = (1.0, "implicit_one") if raw == "" else _parse_coef_token(raw)
            if kind not in ("numeric", "implicit_one"):
                flags.add(kind)
            name = ion[1:-1] if _outer_parens(ion) else ion
            inner_pairs = re.findall(r"([A-Z][a-z]?)(\d*\.?\d*)", name)
            if (_outer_parens(ion) and name not in vocab_set
                    and len(inner_pairs) >= 2 and "".join(a + b for a, b in inner_pairs) == name):
                flags.add("subgroup_expanded")             # Pb(I0.83Br0.17)3
                for el, c in inner_pairs:
                    pairs.append((el, (float(c) if c else 1.0) * val))
            else:
                pairs.append((name, val))
        layers.append(pairs)
    if len(layers) > 1:
        flags.add("multi_layer")
    return layers, flags


# ----------------------------------------------------------------------------------------
# band gap
# ----------------------------------------------------------------------------------------


@lru_cache(maxsize=None)
def _parse_bg_cached(s, lo, hi):
    if s is None:
        return (), frozenset({"missing"})
    flags, vals = set(), []
    for tok in _split_top(s, "|"):
        t = tok.strip()
        low = t.lower()
        if low in MISSING_WORDS:
            vals.append(NAN)
            flags.add("nan_layer")
            continue
        if low in UNKNOWN_WORDS:
            vals.append(NAN)
            flags.add("unknown")
            continue
        t2 = t.replace(",", ".")                       # decimal comma, if any
        m = _RE_RANGE.match(t2)
        if m:
            vals.append(round((float(m.group(1)) + float(m.group(2))) / 2, 6))
            flags.add("range_midpoint")                # '1.8-2.3' -> 2.05
        elif _RE_PLAIN.match(t2):
            vals.append(float(t2))
        else:
            vals.append(NAN)
            flags.add("text_value")
    if len(vals) > 1:
        flags.add("multi_layer")
    if any(np.isfinite(v) and not (lo <= v <= hi) for v in vals):
        flags.add("outside_typical_range")             # informational
    return tuple(vals), frozenset(flags)


def parse_band_gap(value, lo=1.0, hi=3.5):
    """'1.63 | nan' -> ([1.63, nan], flags)   |   1.8-2.3 -> ([2.05], {'range_midpoint'})"""
    s = _to_str(value)
    # floats arrive as 1.3 / 1.6 etc. -> go through str() first (done in _to_str)
    vals, flags = _parse_bg_cached(s, lo, hi)
    return list(vals), set(flags)


# ----------------------------------------------------------------------------------------
# master function
# ----------------------------------------------------------------------------------------


def _join(flags):
    return ";".join(sorted(flags))


def parse_perovskite_columns(df: pd.DataFrame, cols: dict = COLS) -> pd.DataFrame:
    """Return a copy of df with, for each of the 9 columns, two new columns:
         <col>__parsed   python lists (layers -> items)
         <col>__flags    ';'-joined abnormalities found while parsing
       plus band_gap__first / band_gap__mean / n_layers helper columns."""
    out = df.copy()
    vocab = build_ion_vocab(df, [cols["a"], cols["b"], cols["c"]])
    vocab_set = set(vocab)
    tok_re, long_re = _make_regexes(vocab)

    parsed_ions, parsed_coefs = {}, {}
    for site, ic, cc in (("A", cols["a"], cols["ac"]),
                         ("B", cols["b"], cols["bc"]),
                         ("C", cols["c"], cols["cc"])):
        ion_res = [parse_ions(v, site) for v in df[ic]]
        coef_res = [parse_coefficients(v, site) for v in df[cc]]
        parsed_ions[site] = [r[0] for r in ion_res]
        parsed_coefs[site] = [r[0] for r in coef_res]
        ion_flags = [r[1] for r in ion_res]
        coef_flags = [r[1] for r in coef_res]
        for i in range(len(df)):                       # cross-column rules
            coef_flags[i] |= check_site(parsed_ions[site][i], parsed_coefs[site][i], site)
        out[f"{ic}__parsed"] = parsed_ions[site]
        out[f"{ic}__flags"] = [_join(f) for f in ion_flags]
        out[f"{cc}__parsed"] = parsed_coefs[site]
        out[f"{cc}__flags"] = [_join(f) for f in coef_flags]

    # short form: it is DERIVED from the ion columns, so rebuild it and compare.
    # If it matches, the tokens come straight from the ion columns (no guessing);
    # otherwise fall back to the vocabulary tokenizer and flag the row.
    sf_res = [parse_short_form(v, tok_re) for v in df[cols["short"]]]
    sf_layers = [r[0] for r in sf_res]
    sf_flags = [r[1] for r in sf_res]
    for i in range(len(df)):
        raw = _to_str(df[cols["short"]].iat[i])
        if raw is None:
            continue
        tokens = [[x for site in "ABC" if li < len(parsed_ions[site][i]) for x in parsed_ions[site][i][li] if x]
                  for li in range(max(len(parsed_ions[s][i]) for s in "ABC"))]
        rec = [_norm_formula("".join(t)) for t in tokens]
        got = [_norm_formula(l) for l in _split_top(raw, "|")]
        if rec and got == rec:
            sf_layers[i] = tokens
            sf_flags[i].discard("short_form_unparsed_chars")
        elif rec:
            sf_flags[i].add("short_form_differs_from_ion_columns")
    out[f"{cols['short']}__parsed"] = sf_layers
    out[f"{cols['short']}__flags"] = [_join(f) for f in sf_flags]

    # long form (+ ion-name / coefficient check against the ion columns)
    lf_res = [parse_long_form(v, long_re, vocab_set) for v in df[cols["long"]]]
    lf_flags = [r[1] for r in lf_res]
    for i, (layers, _) in enumerate(lf_res):
        if not layers or not parsed_ions["A"][i] and not parsed_ions["B"][i] and not parsed_ions["C"][i]:
            continue
        expect = []
        for li in range(max(len(parsed_ions[s][i]) for s in "ABC")):
            d = {}
            for s in "ABC":
                if li < len(parsed_ions[s][i]):
                    cs = parsed_coefs[s][i][li] if li < len(parsed_coefs[s][i]) else []
                    for k, ion in enumerate(parsed_ions[s][i][li]):
                        if ion:
                            d[ion] = cs[k] if k < len(cs) else NAN
            expect.append(d)
        if len(expect) != len(layers):
            lf_flags[i].add("long_form_layer_count_differs")
            continue
        for d, pairs in zip(expect, layers):
            if sorted(d) != sorted(p[0] for p in pairs):
                lf_flags[i].add("long_form_ions_differ_from_ion_columns")
            else:
                for ion, val in pairs:
                    if np.isfinite(val) and np.isfinite(d[ion]) and not math.isclose(val, d[ion], rel_tol=2e-3, abs_tol=1e-6):
                        lf_flags[i].add("long_form_coefficient_differs")
    out[f"{cols['long']}__parsed"] = [r[0] for r in lf_res]
    out[f"{cols['long']}__flags"] = [_join(f) for f in lf_flags]

    # band gap
    bg_res = [parse_band_gap(v) for v in df[cols["bg"]]]
    out[f"{cols['bg']}__parsed"] = [r[0] for r in bg_res]
    out[f"{cols['bg']}__flags"] = [_join(r[1]) for r in bg_res]
    out["band_gap__first"] = [next((v for v in r[0] if np.isfinite(v)), NAN) for r in bg_res]
    out["band_gap__mean"] = [float(np.nanmean(r[0])) if any(np.isfinite(v) for v in r[0]) else NAN for r in bg_res]
    out["n_layers"] = [max(len(parsed_ions[s][i]) for s in "ABC") for i in range(len(df))]
    return out


def flag_report(parsed: pd.DataFrame) -> pd.DataFrame:
    """Count of every flag per source column - use it to see what is still dirty."""
    rows = []
    for col in [c for c in parsed.columns if c.endswith("__flags")]:
        counts = parsed[col].str.split(";").explode().value_counts()
        for flag, n in counts.items():
            if flag:
                rows.append((col.replace("__flags", ""), flag, n))
    return pd.DataFrame(rows, columns=["column", "flag", "rows"]).sort_values(["column", "rows"], ascending=[True, False])


if __name__ == "__main__":
    demo = pd.DataFrame({
        COLS["short"]: ["MAPbI", "CsFAMAPbBrI", "MAPbI | CsPbI", "nanBiO"],
        COLS["long"]: ["MAPbI3", "Cs0.05FA0.79MA0.16PbBr0.5I2.5", "MAPbI3 | CsPbI3", "nanBiO3"],
        COLS["a"]: ["MA", "Cs; FA; MA", "MA | Cs", np.nan],
        COLS["ac"]: ["1", "0.05; 79; 0.16", "1 | 1", np.nan],
        COLS["b"]: ["Pb", "Pb", "Pb | Pb", "Bi"],
        COLS["bc"]: ["1", "1", "1 | 1", "1"],
        COLS["c"]: ["I", "Br; I", "I | I", "O"],
        COLS["cc"]: ["3", "0.5; 2.5", "3 | 3", "3"],
        COLS["bg"]: [1.6, "1.8-2.3", "1.5 | 1.7", np.nan],
    })
    res = parse_perovskite_columns(copy_df)
    print(res.filter(like="__").T.to_string())
    print(flag_report(res))


                                                                                  0                                   1                                   2                                   3                                   4                                   5                                   6                                   7                                   8                                   9                                   10                                  11                                  12                                  13                                  14                                  15                                  16                                  17                                  18                                  19                                  20                                  21                                  22                                  23                                  24                                  25                


### Done

* `Perovskite_composition_short_form`
* `Perovskite_composition_long_form`
* `Perovskite_composition_a_ions` / `Perovskite_composition_a_ions_coefficients`
* `Perovskite_composition_b_ions` / `Perovskite_composition_b_ions_coefficients`
* `Perovskite_composition_c_ions` / `Perovskite_composition_c_ions_coefficients`
* `Perovskite_band_gap`



Numerical Column Parsing -

In [5]:
"""
perovskite_parser_extra.py
==========================
Parsers for a second batch of Perovskite Database columns (rules from the v5.4 description).

THICKNESS columns   (Perovskite_thickness, ETL_thickness, HTL_thickness, Backcontact_thickness_list)
    "Th.1 | Th.2 | ... | Th.n"  [nm]     layers separated by ' | ', unknown layer = 'nan'
    -> <col>__list   flat list of per-layer thicknesses  (unknown layer -> None)
       <col>__sum    total thickness; NaN as soon as ONE layer is unknown (or the cell is missing)
       <col>__flags

NESTED columns      (solvents_mixing_ratios, thermal_annealing_temperature, thermal_annealing_time)
    "V1; V2 >> V3; V4 | V5 >> ..."        ' | ' = layers, ' >> ' = deposition steps, ';' = values
    -> <col>__parsed   nested list  [layer][step][value]   (string 'nan' -> None)
       <col>__flags    (incl. temperature-vs-time alignment check)

SCALAR columns      (number_of_deposition_steps, cells_per_substrate, Cell_area_measured, PCE, Voc, Jsc, FF)
    -> <col>__parsed   one number per row  (string 'nan' / '' / 'Unknown' -> None)
       <col>__flags

Everything that is odd is reported in the __flags columns, nothing is silently dropped.
Repairs that change a value (FF in %, Voc in mV, ranges -> midpoint, unit conversion) are flagged and
can be switched off with repair=False.

Usage
-----
    from perovskite_parser_extra import parse_more_columns, flag_report
    parsed2 = parse_more_columns(copy_df)                  # or parse_more_columns(parsed) to chain
    print(flag_report(parsed2))
"""
from __future__ import annotations

import math
import re
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------------------
# configuration - edit the column names / kinds here if your dataframe uses other names
# ----------------------------------------------------------------------------------------
THICKNESS_COLS = [
    "Perovskite_thickness",
    "ETL_thickness",
    "HTL_thickness",
    "Backcontact_thickness_list",
]

NESTED_COLS = {                      # column -> kind
    "Perovskite_deposition_solvents_mixing_ratios": "ratio",
    "Perovskite_deposition_thermal_annealing_temperature": "temp",
    "Perovskite_deposition_thermal_annealing_time": "time",
}

SCALAR_COLS = {                      # column -> kind
    "Perovskite_deposition_number_of_deposition_steps": "steps",
    "Cell_number_of_cells_per_substrate": "cells",
    "Cell_area_measured": "area",
    "JV_default_PCE": "pce",
    "Voc": "voc",                    # database name: JV_default_Voc
    "Jsc": "jsc",                    # database name: JV_default_Jsc
    "FF": "ff",                      # database name: JV_default_FF
}

# kind -> (is_integer, typical_low, typical_high)   -> only used for the 'outside_typical_range' flag
SCALAR_SPECS = {
    "steps": (True, 0, 10),
    "cells": (True, 0, 1000),
    "area": (False, 0, 100),        # cm^2
    "pce": (False, 0, 35),          # %
    "voc": (False, 0, 2.5),         # V
    "jsc": (False, 0, 50),          # mA/cm^2
    "ff": (False, 0, 1),            # fraction
}

NAN = float("nan")
MISSING_WORDS = {"", "nan", "none", "null", "n/a", "na", "nat", "<na>"}
UNKNOWN_WORDS = {"unknown", "<unknown>"}

# units that are accepted next to a number: unit -> multiplier to the database unit
UNIT_RULES = {
    "thickness": {"nm": 1, "um": 1e3, "µm": 1e3},
    "time": {"min": 1, "mins": 1, "minute": 1, "minutes": 1,
             "h": 60, "hr": 60, "hrs": 60, "hour": 60, "hours": 60,
             "s": 1 / 60, "sec": 1 / 60, "secs": 1 / 60, "second": 1 / 60, "seconds": 1 / 60},
    "temp": {"c": 1, "°c": 1, "℃": 1, "degc": 1},
    "area": {"cm2": 1, "cm^2": 1, "cm²": 1},
    "pce": {"%": 1},
    "voc": {"v": 1, "mv": 1e-3},
    "jsc": {"ma/cm2": 1, "ma/cm^2": 1, "ma/cm²": 1, "macm-2": 1},
    "ff": {"%": 0.01},
}

# ----------------------------------------------------------------------------------------
# numeric token parser  (shared by every column)
# ----------------------------------------------------------------------------------------
_N = r"[+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?"
_P = r"(?:\d+\.?\d*|\.\d+)"
_RE_NUM_UNIT = re.compile(rf"^(?P<num>{_N})\s*(?P<unit>[A-Za-zµ°℃%][A-Za-zµ°℃%²/^\-\.\s0-9]*)?$")
_RE_RANGE = re.compile(r"^(.+?)(?:\s*[–—~]\s*|(?<=[\d\w%])\s*-\s*(?=[\d.])|\s+to\s+)(.+)$", re.I)
_RE_PLUSMINUS = re.compile(rf"^({_N}[^\d±+]*?)\s*(?:±|\+/-|\+-)\s*{_P}.*$")


def _num_unit(t, unit_kind):
    m = _RE_NUM_UNIT.match(t)
    if not m:
        return None
    v, flags = float(m.group("num")), set()
    u = m.group("unit")
    if u:
        key = re.sub(r"[\s.]", "", u.lower())
        rules = UNIT_RULES.get(unit_kind, {})
        if key not in rules:
            return None                                  # unknown unit -> treated as text
        if rules[key] != 1:
            v *= rules[key]
            flags.add("unit_converted")
    return v, flags


def parse_number(tok, unit_kind=""):
    """One token -> (float | None, flags).  None for every kind of missing / unusable value."""
    t = str(tok).strip()
    low = t.lower()
    if low == "":
        return None, {"empty_token"}
    if low in MISSING_WORDS:
        return None, {"string_nan_to_none"}              # the 'nan' *string* problem
    if low in UNKNOWN_WORDS:
        return None, {"unknown_to_none"}
    if t[0] in "<>≤≥":
        return None, {"inequality"}
    flags = set()
    t2 = re.sub(r"^(?:~|≈|ca\.?\s*|approx\.?\s*)", "", t, flags=re.I).strip()
    if t2 != t:
        flags.add("approx")
    m = _RE_PLUSMINUS.match(t2)
    if m:
        t2 = m.group(1).strip()
        flags.add("plus_minus")
    r = _num_unit(t2, unit_kind)
    if r is None:                                        # 90-110, 90 to 110, 90~110
        m = _RE_RANGE.match(t2)
        if m:
            ra, rb = _num_unit(m.group(1).strip(), unit_kind), _num_unit(m.group(2).strip(), unit_kind)
            if ra and rb:
                r = (round((ra[0] + rb[0]) / 2, 9), ra[1] | rb[1] | {"range_midpoint"})
    if r is None:
        return None, flags | {"text_value"}
    v, f = r
    flags |= f
    if not math.isfinite(v):
        return None, flags | {"non_finite"}
    return v, flags


def _norm(x):
    """Cell -> stripped string, or None for real NaN/None/pd.NA."""
    if x is None:
        return None
    if isinstance(x, str):
        return x.strip()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, (float, np.floating)) and not np.isfinite(x):
        return None
    return str(x)


def _is_missing_cell(s):
    return s is None or s.lower() in MISSING_WORDS


def _join(flags):
    return ";".join(sorted(flags))


def _fill(seq, fill):
    return [fill if v is None else v for v in seq]


# ----------------------------------------------------------------------------------------
# THICKNESS columns
# ----------------------------------------------------------------------------------------
@lru_cache(maxsize=None)
def _thickness_cached(s):
    if _is_missing_cell(s):
        return None, NAN, frozenset({"missing"} | ({"string_nan_to_none"} if s is not None else set()))
    flags, vals = set(), []
    if ";" in s:
        flags.add("unexpected_semicolon")                 # only ' | ' is a layer separator in the rules
    for part in re.split(r"\s*[|;]\s*", s):
        v, f = parse_number(part, "thickness")
        flags |= f
        if v is not None:
            if v <= 0:
                flags.add("non_positive")
            elif v > 1e4:
                flags.add("suspicious_magnitude")         # > 10 um: wrong unit?
        vals.append(v)
    if len(vals) > 1:
        flags.add("multi_layer")
    total = NAN if any(v is None for v in vals) else float(sum(vals))
    if any(v is None for v in vals):
        flags.add("sum_is_nan_unknown_layer")
    return tuple(vals), total, frozenset(flags)


def parse_thickness(value):
    """'20 | nan | 150' -> ([20.0, None, 150.0], nan, flags)   |   '20 | 150' -> ([20.0, 150.0], 170.0, flags)"""
    vals, total, flags = _thickness_cached(_norm(value))
    return (None if vals is None else list(vals)), total, set(flags)


# ----------------------------------------------------------------------------------------
# NESTED columns  (layers | steps >> values ;)
# ----------------------------------------------------------------------------------------
_NESTED_CHECKS = {
    "temp": lambda v: "outside_typical_range" if (v < -50 or v > 800) else None,
    "time": lambda v: "negative" if v < 0 else ("suspicious_magnitude" if v > 1e4 else None),
    "ratio": lambda v: "negative" if v < 0 else None,
}


@lru_cache(maxsize=None)
def _nested_cached(s, kind):
    if _is_missing_cell(s):
        return None, frozenset({"missing"} | ({"string_nan_to_none"} if s is not None else set()))
    flags, layers = set(), []
    check = _NESTED_CHECKS.get(kind)
    for lay in re.split(r"\s*\|\s*", s):
        steps = []
        for st in re.split(r"\s*>>\s*", lay):
            if re.search(r",\s+", st):
                flags.add("comma_instead_of_semicolon")
            toks = re.split(r"\s*;\s*|,\s+", st)
            if kind == "ratio" and any(":" in t for t in toks):
                flags.add("colon_ratio")
                toks = [x for t in toks for x in t.split(":")]
            vals = []
            for t in toks:
                v, f = parse_number(t, kind)
                flags |= f
                if v is not None and check and check(v):
                    flags.add(check(v))
                vals.append(v)
            steps.append(tuple(vals))
        layers.append(tuple(steps))
    if len(layers) > 1:
        flags.add("multi_layer")
    if any(len(l) > 1 for l in layers):
        flags.add("multi_step")
    if any(len(st) > 1 for l in layers for st in l):
        flags.add("multi_value")
    return tuple(layers), frozenset(flags)


def parse_nested(value, kind):
    """'25 >> 100; 150 | 100' -> [[[25.0],[100.0,150.0]],[[100.0]]]   (nan strings -> None)"""
    layers, flags = _nested_cached(_norm(value), kind)
    if layers is None:
        return None, set(flags)
    return [[list(st) for st in lay] for lay in layers], set(flags)


def check_alignment(temp, time):
    """PDF: annealing times must line up with temperatures in layers, steps and values."""
    flags = set()
    if temp is None or time is None:
        return flags
    if len(temp) != len(time):
        return {"layer_count_mismatch_temp_vs_time"}
    for tl, ml in zip(temp, time):
        if len(tl) != len(ml):
            flags.add("step_count_mismatch_temp_vs_time")
            continue
        for ts, ms in zip(tl, ml):
            if len(ts) != len(ms) and not (len(ms) == 1 and ms[0] is None):   # time 'nan' = unknown program
                flags.add("value_count_mismatch_temp_vs_time")
    return flags


# ----------------------------------------------------------------------------------------
# SCALAR columns
# ----------------------------------------------------------------------------------------
@lru_cache(maxsize=None)
def _scalar_cached(s, kind, repair):
    is_int, lo, hi = SCALAR_SPECS[kind]
    if _is_missing_cell(s):
        return None, frozenset({"missing"} | ({"string_nan_to_none"} if s is not None else set()))
    v, flags = parse_number(s, kind)
    if v is None:
        return None, frozenset(flags)
    if kind == "ff" and 10 <= v <= 100:                             # FF given in % (1<v<10 is just flagged)
        flags.add("percent_to_fraction")
        if repair:
            v /= 100
    if kind == "voc" and 100 <= v <= 3000:                          # Voc given in mV
        flags.add("mV_to_V")
        if repair:
            v /= 1000
    if is_int:
        if abs(v - round(v)) < 1e-9:
            v = int(round(v))
        else:
            return None, frozenset(flags | {"non_integer"})
        if v < 0:
            flags.add("negative")
        if v == 0:
            flags.add("zero_default_placeholder")                   # database default is 0 = 'not stated'
    elif kind == "area" and v <= 0:
        flags.add("non_positive")
    if not (lo <= v <= hi) and not (is_int and v == 0):
        flags.add("outside_typical_range")
    return v, frozenset(flags)


def parse_scalar(value, kind, repair=True):
    """'nan' -> (None, {...});  '0.95' -> (0.95, set());  '85' for FF -> (0.85, {'percent_to_fraction'})"""
    v, flags = _scalar_cached(_norm(value), kind, repair)
    return v, set(flags)


# ----------------------------------------------------------------------------------------
# master function
# ----------------------------------------------------------------------------------------
def parse_more_columns(df, thickness_cols=THICKNESS_COLS, nested_cols=NESTED_COLS,
                       scalar_cols=SCALAR_COLS, repair=True, scalar_missing="none", list_missing=None):
    """
    scalar_missing : 'none' -> scalar parsed columns are object dtype, missing = Python None (as requested)
                     'nan'  -> float64 (NaN) for float columns, nullable Int64 (<NA>) for integer columns
    list_missing   : what an unknown layer/value looks like INSIDE the lists: None (default) or float('nan')
    repair         : False -> FF in %, Voc in mV are only flagged, not rescaled
    """
    idx, new = df.index, {}

    def _col_ok(c):
        if c in df.columns:
            return True
        warnings.warn(f"column '{c}' not found - skipped")
        return False

    for col in filter(_col_ok, thickness_cols):
        res = [parse_thickness(v) for v in df[col]]
        new[f"{col}__list"] = pd.Series([None if r[0] is None else _fill(r[0], list_missing) for r in res],
                                        index=idx, dtype=object)
        new[f"{col}__sum"] = pd.Series([r[1] for r in res], index=idx, dtype="float64")
        new[f"{col}__flags"] = pd.Series([_join(r[2]) for r in res], index=idx)

    nested = {}
    for col, kind in filter(lambda kv: _col_ok(kv[0]), nested_cols.items()):
        res = [parse_nested(v, kind) for v in df[col]]
        nested[kind] = ([r[0] for r in res], [r[1] for r in res], col)
    if "temp" in nested and "time" in nested:                      # cross-column alignment rule
        tp, tf, _ = nested["temp"]
        ti, tif, _ = nested["time"]
        for i in range(len(df)):
            fl = check_alignment(tp[i], ti[i])
            tf[i] |= fl
            tif[i] |= fl
    for kind, (parsed, flags, col) in nested.items():
        parsed = [None if p is None else [[_fill(st, list_missing) for st in lay] for lay in p] for p in parsed]
        new[f"{col}__parsed"] = pd.Series(parsed, index=idx, dtype=object)
        new[f"{col}__flags"] = pd.Series([_join(f) for f in flags], index=idx)

    for col, kind in filter(lambda kv: _col_ok(kv[0]), scalar_cols.items()):
        res = [parse_scalar(v, kind, repair) for v in df[col]]
        vals = [r[0] for r in res]
        s = pd.Series(vals, index=idx, dtype=object)
        if scalar_missing == "nan":
            s = pd.Series(pd.array(vals, dtype="Int64"), index=idx) if SCALAR_SPECS[kind][0] \
                else s.astype("float64")
        new[f"{col}__parsed"] = s
        new[f"{col}__flags"] = pd.Series([_join(r[1]) for r in res], index=idx)

    return pd.concat([df, pd.DataFrame(new, index=idx)], axis=1)


def flag_report(parsed: pd.DataFrame) -> pd.DataFrame:
    """Count of every flag per source column - shows what is still dirty."""
    rows = []
    for col in [c for c in parsed.columns if c.endswith("__flags")]:
        counts = parsed[col].astype(str).str.split(";").explode().value_counts()
        for flag, n in counts.items():
            if flag:
                rows.append((col.replace("__flags", ""), flag, n))
    return pd.DataFrame(rows, columns=["column", "flag", "rows"]).sort_values(
        ["column", "rows"], ascending=[True, False]).reset_index(drop=True)


if __name__ == "__main__":
    demo = pd.DataFrame({
        "Perovskite_thickness": ["450", "20 | nan | 150", np.nan, "300-500", "nan"],
        "ETL_thickness": ["30 | 150", "nan | nan", "50 nm", "0.5 um", "60;120"],
        "HTL_thickness": ["200", "nan", "nan | 100", "~150", np.nan],
        "Backcontact_thickness_list": ["100", "80", "nan", "100 | 5", np.nan],
        "Perovskite_deposition_solvents_mixing_ratios": ["1", "4; 1", "0.8; 0.2 >> 1", "nan", "1 | 1"],
        "Perovskite_deposition_thermal_annealing_temperature": ["100", "25; 100", "70 >> 100", "nan", "100 | 25"],
        "Perovskite_deposition_thermal_annealing_time": ["10", "nan", "2 >> 10; 5", "nan", "10 | nan"],
        "Perovskite_deposition_number_of_deposition_steps": ["1", "2", "nan", "0", np.nan],
        "Cell_number_of_cells_per_substrate": ["6", "nan", "0", "8.0", "2.5"],
        "Cell_area_measured": ["0.09", "nan", "0.1 cm2", "0.04-0.06", np.nan],
        "JV_default_PCE": ["18.5", "nan", "19.1%", "-3", "120"],
        "Voc": ["1.05", "1050", "nan", "Unknown", np.nan],
        "Jsc": ["22.3", "nan", "0.0223", "60", np.nan],
        "FF": ["0.78", "78", "nan", "1.5", "0,78"],
    })
    res = parse_more_columns(copy_df)
    pd.set_option("display.width", 250, "display.max_colwidth", 60)
    print(res.filter(like="__").T.to_string())
    print(flag_report(res).to_string())



                                                                                                                                    0                                                        1                                                        2                                                        3                                                        4                                                        5                                                        6                                                        7                         8                                                        9                         10                        11                        12                        13                                                       14                                                       15                                                       16                                                       17                                                    

In [6]:
parsed2 = parse_more_columns(copy_df)

In [7]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)

scalar_cols = [
    "Perovskite_deposition_number_of_deposition_steps",
    "Cell_number_of_cells_per_substrate",
    "Cell_area_measured",
    "JV_default_PCE",
    "Voc",
    "Jsc",
    "FF"
]

thick_cols = [
    "Perovskite_thickness",
    "ETL_thickness",
    "HTL_thickness",
    "Backcontact_thickness_list"
]

nested_cols = [
    "Perovskite_deposition_solvents_mixing_ratios",
    "Perovskite_deposition_thermal_annealing_temperature",
    "Perovskite_deposition_thermal_annealing_time"
]

***`Perovskite_deposition_synthesis_atmosphere`*** parsing

In [8]:
import re
import pandas as pd

# ============================================================
# COLUMN
# ============================================================

column = "Perovskite_deposition_synthesis_atmosphere"


# ============================================================
# REMOVE OLD DIAGNOSTIC COLUMNS FROM PREVIOUS ATTEMPTS
# ============================================================

old_columns = [
    f"{column}_clean",
    f"{column}_parsed",
    f"{column}_flags",
    f"{column}_non_atmosphere_tokens"
]

copy_df = copy_df.drop(
    columns=[c for c in old_columns if c in copy_df.columns]
)


# ============================================================
# 1. CONFIRMED TERMINOLOGY CORRECTIONS
# ============================================================

atmosphere_replacements = {
    "Ait": "Air",
    "Ari": "Air",
    "MaBr": "MABr",
    "Pbl2": "PbI2",
    "Hi": "HI",
    "toluene": "Toluene",
    "Methyl amine": "Methylamine",
    "Methylamin": "Methylamine",
}


def clean_atmosphere(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    for old, new in atmosphere_replacements.items():

        value = re.sub(
            rf"(?<!\w){re.escape(old)}(?!\w)",
            new,
            value
        )

    return value


copy_df[f"{column}_clean"] = copy_df[column].apply(
    clean_atmosphere
)


# ============================================================
# 2. PARSER
#
# Structure:
#
# Layer 1 | Layer 2
#
# Each layer:
#
# Step 1 >> Step 2
#
# Each step:
#
# Gas 1; Gas 2
# ============================================================

def parse_atmosphere(value):

    # --------------------------------------------------------
    # Missing
    # --------------------------------------------------------

    if pd.isna(value):
        return [], {"missing"}

    value = str(value).strip()

    if not value:
        return [], {"empty"}

    # --------------------------------------------------------
    # Exact Unknown = missing
    # --------------------------------------------------------

    if value == "Unknown":
        return [], {"missing", "unknown"}

    flags = set()
    parsed_layers = []

    # --------------------------------------------------------
    # Layer level: |
    # --------------------------------------------------------

    for layer in value.split("|"):

        layer = layer.strip()

        if not layer:
            flags.add("empty_layer")
            continue

        parsed_steps = []

        # ----------------------------------------------------
        # Step level: >>
        # ----------------------------------------------------

        for step in layer.split(">>"):

            step = step.strip()

            if not step:
                flags.add("empty_step")
                continue

            parsed_gases = []

            # ------------------------------------------------
            # Gas level: ;
            # ------------------------------------------------

            for gas in step.split(";"):

                gas = gas.strip()

                if not gas:
                    flags.add("empty_gas")
                    continue

                # Embedded Unknown = missing gas information
                if gas == "Unknown":

                    parsed_gases.append(None)
                    flags.add("embedded_missing")

                else:

                    parsed_gases.append(gas)

            if parsed_gases:
                parsed_steps.append(parsed_gases)

        if parsed_steps:
            parsed_layers.append(parsed_steps)

    # ========================================================
    # STRUCTURAL FLAGS
    # ========================================================

    # More than one layer
    if len(parsed_layers) > 1:
        flags.add("multiple_layers")

    # More than one sequential step within a layer
    if any(
        len(layer) > 1
        for layer in parsed_layers
    ):
        flags.add("multiple_steps")

    # More than one gas in a single step
    if any(
        len(step) > 1
        for layer in parsed_layers
        for step in layer
    ):
        flags.add("mixed_gases")

    # ========================================================
    # SUSPICIOUS COMPOSITION-LIKE VALUES
    # ========================================================

    if re.search(
        r"@|\b25\b|\b50\b|\b75\b",
        value
    ):
        flags.add("composition_like")

    return parsed_layers, flags


# ============================================================
# 3. APPLY PARSER
# ============================================================

parsed_atmosphere = copy_df[
    f"{column}_clean"
].apply(parse_atmosphere)


copy_df[f"{column}_parsed"] = parsed_atmosphere.apply(
    lambda x: x[0]
)


copy_df[f"{column}_flags"] = parsed_atmosphere.apply(
    lambda x: ";".join(sorted(x[1]))
    if x[1]
    else pd.NA
)


# ============================================================
# 4. CLEANING SUMMARY
# ============================================================

changed = (
    copy_df[column].astype("string")
    != copy_df[f"{column}_clean"].astype("string")
)


print("=" * 100)
print(f"{column} — CLEANING + PARSING COMPLETE")
print("=" * 100)

print("Total rows:", len(copy_df))
print("Original missing:", copy_df[column].isna().sum())
print("Changed rows:", changed.sum())
print(
    "Cleaned missing:",
    copy_df[f"{column}_clean"].isna().sum()
)


# ============================================================
# 5. FLAG SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("FLAG SUMMARY")
print("=" * 100)

flag_counts = {}

for flags in copy_df[f"{column}_flags"]:

    if pd.isna(flags):
        continue

    for flag in str(flags).split(";"):

        if flag:
            flag_counts[flag] = flag_counts.get(flag, 0) + 1


if flag_counts:

    flag_summary = (
        pd.Series(flag_counts)
        .sort_values(ascending=False)
    )

    print(flag_summary)

else:

    print("No flags found.")


# ============================================================
# 6. CONFIRMED TERMINOLOGY CHANGES
# ============================================================

print("\n" + "=" * 100)
print("TERMINOLOGY CHANGES")
print("=" * 100)

if changed.sum() > 0:

    display(
        copy_df.loc[
            changed,
            [
                column,
                f"{column}_clean"
            ]
        ]
        .drop_duplicates()
        .sort_values(column)
    )

else:

    print("No terminology changes.")


# ============================================================
# 7. PARSED EXAMPLES
# ============================================================

print("\n" + "=" * 100)
print("PARSED EXAMPLES")
print("=" * 100)

display(
    copy_df[
        [
            column,
            f"{column}_clean",
            f"{column}_parsed",
            f"{column}_flags"
        ]
    ].head(30)
)


# ============================================================
# 8. INSPECT ALL STRUCTURALLY FLAGGED ROWS
# ============================================================

print("\n" + "=" * 100)
print("STRUCTURALLY FLAGGED EXAMPLES")
print("=" * 100)

flagged_mask = copy_df[f"{column}_flags"].notna()

display(
    copy_df.loc[
        flagged_mask,
        [
            column,
            f"{column}_clean",
            f"{column}_parsed",
            f"{column}_flags"
        ]
    ].head(50)
)

Perovskite_deposition_synthesis_atmosphere — CLEANING + PARSING COMPLETE
Total rows: 49027
Original missing: 2901
Changed rows: 147
Cleaned missing: 2901

FLAG SUMMARY
missing             17676
unknown             14775
multiple_steps       8253
mixed_gases           716
embedded_missing      323
multiple_layers        96
composition_like        3
dtype: int64

TERMINOLOGY CHANGES


,Perovskite_deposition_synthesis_atmosphere,Perovskite_deposition_synthesis_atmosphere_clean
32728,Air >> Air; HI >> Air; Methylamin >> Air; Hi,Air >> Air; HI >> Air; Methylamine >> Air; HI
5225,Air >> Air; Methylamin,Air >> Air; Methylamine
13660,Air >> Ar; Methylamin,Air >> Ar; Methylamine
4412,Air >> MaBr,Air >> MABr
20307,Air >> Methyl amine,Air >> Methylamine
2192,Air >> Methylamin; Vacuum,Air >> Methylamine; Vacuum
35952,Air | Ari; MAI,Air | Air; MAI
6547,Ait >> Air; MAI,Air >> Air; MAI
5163,Ar; MAI; Pbl2,Ar; MAI; PbI2
5474,Dry air >> Dry air; Methylamin,Dry air >> Dry air; Methylamine



PARSED EXAMPLES


,Perovskite_deposition_synthesis_atmosphere,Perovskite_deposition_synthesis_atmosphere_clean,Perovskite_deposition_synthesis_atmosphere_parsed,Perovskite_deposition_synthesis_atmosphere_flags
0,Unknown,Unknown,[],missing;unknown
1,Unknown,Unknown,[],missing;unknown
2,Unknown,Unknown,[],missing;unknown
3,Unknown,Unknown,[],missing;unknown
4,Unknown,Unknown,[],missing;unknown
5,Unknown,Unknown,[],missing;unknown
6,Unknown,Unknown,[],missing;unknown
7,Unknown,Unknown,[],missing;unknown
8,Air,Air,[[[Air]]],<NA>
9,Unknown,Unknown,[],missing;unknown



STRUCTURALLY FLAGGED EXAMPLES


,Perovskite_deposition_synthesis_atmosphere,Perovskite_deposition_synthesis_atmosphere_clean,Perovskite_deposition_synthesis_atmosphere_parsed,Perovskite_deposition_synthesis_atmosphere_flags
0,Unknown,Unknown,[],missing;unknown
1,Unknown,Unknown,[],missing;unknown
2,Unknown,Unknown,[],missing;unknown
3,Unknown,Unknown,[],missing;unknown
4,Unknown,Unknown,[],missing;unknown
5,Unknown,Unknown,[],missing;unknown
6,Unknown,Unknown,[],missing;unknown
7,Unknown,Unknown,[],missing;unknown
9,Unknown,Unknown,[],missing;unknown
10,Unknown,Unknown,[],missing;unknown


***`Perovskite_deposition_solvents`*** parsing

In [9]:
import pandas as pd

# ============================================================
# COLUMN
# ============================================================

solvent_col = "Perovskite_deposition_solvents"


# ============================================================
# KEEP ORIGINAL VALUES
# ============================================================

original_solvents = copy_df[solvent_col].copy()


# ============================================================
# CANONICALIZATION MAP
# ============================================================

solvent_replacements = {

    # Water
    "H2O": "Water",

    # Alcohols
    "MeOH": "Methanol",
    "EtOH": "Ethanol",
    "IPA": "Isopropanol",
    "2-propanol": "Isopropanol",
    "isopropanol": "Isopropanol",
    "isopropyl alcohol": "Isopropanol",

    # Methoxyethanol
    "2-ME": "2-methoxyethanol",
    "2ME": "2-methoxyethanol",
    "2-Methoxy-ethanol": "2-methoxyethanol",

    # Acetonitrile
    "ACN": "Acetonitrile",
    "acetonitrile": "Acetonitrile",

    # GBL
    "GLB": "GBL",
    "γ-butyrolactone": "GBL",
    "gamma-butyrolactone": "GBL",
    "γ-Butyrolactone": "GBL",

    # Esters
    "EtOAc": "Ethyl acetate",
    "MeOAc": "Methyl acetate",

    # NMP
    "N-Methyl-2-pyrrolidone": "NMP",
    "N-methyl-2-pyrrolidone": "NMP",
    "n-methyl-2-pyrrolidone": "NMP",
    "N-methyl-2-pyrrolidinone": "NMP",
    "1-Methyl-2-pyrrolidinone": "NMP",

    # DMF
    "N,N-dimethylformamide": "DMF",
    "dimethylformamide": "DMF",
    "N-Dimethylformamide": "DMF",
    "N,N′-dimethylformamide": "DMF",
    "N'N-dimethylformamide": "DMF",

    # DMSO
    "dimethyl sulfoxide": "DMSO",
    "dimethylsulfoxide": "DMSO",

    # DMAc
    "Dimetyletanamid": "DMAc",
    "Dimethylacetamide": "DMAc",
    "N,N-dimethylacetamide": "DMAc",

    # Hexane
    "n-hexane": "Hexane",

    # Pyridine
    "Pyridin": "Pyridine",
    "pyridine": "Pyridine",

    # Acetone
    "Aceton": "Acetone",
    "acetone": "Acetone",

    # Butanol
    "n-butanol": "Butanol",
    "n-Butanol": "Butanol",
    "1-butanol": "Butanol",
    "1-Butanol": "Butanol",
    "butanol": "Butanol",
    "n-butyl alchohol": "Butanol",

    # Capitalization
    "chlorobenzene": "Chlorobenzene",
    "toluene": "Toluene",
    "anisole": "Anisole",

    # Clear spelling corrections
    "Formarmid": "Formamide",
    "Hexame": "Hexane",
    "Methoxyactonitrile": "Methoxyacetonitrile",
    "Methybenene": "Toluene",
    "Methylfomramid": "Methylformamide",
    "n-hexyl alchohol": "n-hexanol",
    "Tetrahydrothiophene 1-oxid": "Tetrahydrothiophene-1-oxide",
    "g-Valerolacetone": "γ-valerolactone",

    # Clear capitalization variants
    "ethanol": "Ethanol",
    "formamide": "Formamide",
    "glycerol": "Glycerol",
    "methanol": "Methanol",
    "methylamine": "Methylamine",
    "water": "Water",
    "chloroform": "Chloroform",

    # Clearly equivalent naming
    "1-pentanol": "Pentan-1-ol",
    "n-Heptane": "Heptane",
    "n-amyl alcohol": "Pentan-1-ol",
    "tert-butanol": "Tert-butanol",
    "γ-butyrolactone (GBL)": "GBL",

     # Clear spelling / typo corrections
    "2‐butoxyethano": "2-butoxyethanol",
    "Ocatane": "Octane",

    # Clear capitalization / naming variants
    "Isopropyl alcohol": "Isopropanol",
    "anhydrous isopropanol": "Anhydrous isopropanol",
    "anhydrous methanol": "Anhydrous methanol",

    # Clear equivalent names
    "Methylacetate": "Methyl acetate",
    "n-butylamine": "n-butylamine",

    # Obvious abbreviation expansion
    "o-DCB": "1,2-dichlorobenzene",
    "o-dichlorobenzene": "1,2-dichlorobenzene",
    "chloronaphthalene": "1-Chloronaphthalene",

}

# ============================================================
# BASIC CLEANING
# ============================================================

copy_df[solvent_col] = copy_df[solvent_col].astype("string")

copy_df[solvent_col] = (
    copy_df[solvent_col]
    .str.strip()
)


# ============================================================
# MISSING VALUES
# ============================================================

copy_df[solvent_col] = copy_df[solvent_col].replace({
    "Unknown": pd.NA,
    "unknown": pd.NA,
    "UNKNOWN": pd.NA,
    "NaN": pd.NA,
    "nan": pd.NA,
    "NAN": pd.NA,
})


# ============================================================
# APPLY CANONICALIZATION
# ============================================================

copy_df[solvent_col] = copy_df[solvent_col].replace(
    solvent_replacements
)


# ============================================================
# IDENTIFY CHANGED VALUES
# ============================================================

changed = pd.DataFrame({
    "Perovskite_deposition_solvents_original": original_solvents,
    "Perovskite_deposition_solvents_clean": copy_df[solvent_col]
})

changed = changed[
    changed[
        "Perovskite_deposition_solvents_original"
    ].astype("string")
    != changed[
        "Perovskite_deposition_solvents_clean"
    ].astype("string")
]


# ============================================================
# DISPLAY CHANGES
# ============================================================

print("=" * 100)
print("SOLVENT VALUES THAT CHANGED")
print("=" * 100)

print(
    changed.to_string(index=False)
)

# ============================================================
# STRUCTURAL PARSING
# ============================================================

import re
import pandas as pd

# ============================================================
# 1. REMOVE EMBEDDED MIXING-RATIO ANNOTATIONS
#    FOR STRUCTURAL PARSING ONLY
# ============================================================

def remove_embedded_ratio(value):

    if pd.isna(value):
        return value

    value = str(value)

    # Example:
    # DMSO; GBL @ 4; 7 >> IPA
    # becomes:
    # DMSO; GBL >> IPA

    value = re.sub(
        r"\s*@\s*[^|>]+(?=\s*(?:>>|\||$))",
        "",
        value
    )

    return value


copy_df["Perovskite_deposition_solvents__structure_input"] = (
    copy_df[solvent_col].apply(remove_embedded_ratio)
)


# ============================================================
# 2. PARSE SOLVENT STRUCTURE
# ============================================================

def parse_solvent_structure(value):

    if pd.isna(value):
        return [], {"missing"}

    raw = str(value).strip()

    if not raw:
        return [], {"empty_value"}

    flags = set()
    blocks = []

    # | = separate blocks
    for block in raw.split("|"):

        block = block.strip()

        if not block:
            flags.add("empty_block")
            blocks.append([])
            continue

        steps = []

        # >> = sequential steps
        for step in block.split(">> "):

            step = step.strip()

            if not step:
                flags.add("empty_step")
                continue

            solvents = []

            # ; = solvent components
            for solvent in step.split(";"):

                solvent = solvent.strip()

                if not solvent:
                    flags.add("empty_solvent")
                    continue

                solvents.append(solvent)

            steps.append(solvents)

        blocks.append(steps)

    # Structural flags
    if len(blocks) > 1:
        flags.add("multiple_blocks")

    if any(len(block) > 1 for block in blocks):
        flags.add("multiple_steps")

    if any(
        len(step) > 1
        for block in blocks
        for step in block
    ):
        flags.add("multiple_solvents")

    if raw.startswith("|") or raw.endswith("|"):
        flags.add("leading_or_trailing_block_separator")

    if raw.startswith(">> ") or raw.endswith(">> "):
        flags.add("leading_or_trailing_step_separator")

    if raw.startswith(";") or raw.endswith(";"):
        flags.add("leading_or_trailing_solvent_separator")

    if "||" in raw:
        flags.add("empty_block")

    if ">> >> " in raw:
        flags.add("empty_step")

    if ";;" in raw:
        flags.add("empty_solvent")

    # Preserve these notations
    if ":" in raw:
        flags.add("contains_colon")

    if "/" in raw:
        flags.add("contains_slash")

    if "," in raw:
        flags.add("contains_comma")

    return blocks, flags


parsed_solvents = copy_df[
    "Perovskite_deposition_solvents__structure_input"
].apply(parse_solvent_structure)


copy_df["Perovskite_deposition_solvents__parsed"] = (
    parsed_solvents.apply(lambda x: x[0])
)

copy_df["Perovskite_deposition_solvents__flags"] = (
    parsed_solvents.apply(
        lambda x: ";".join(sorted(x[1]))
    )
)


# ============================================================
# 3. CLEAN THE PARSED COMPONENTS
# ============================================================

def clean_solvent_component(component):

    component = component.strip()

    if not component:
        return component

    # Missing values inside a structured expression
    if component.lower() in {
        "unknown",
        "none",
        "nan"
    }:
        return pd.NA

    # Apply the same approved canonicalization map
    if component in solvent_replacements:
        return solvent_replacements[component]

    return component


def clean_parsed_solvents(blocks):

    cleaned_blocks = []

    for block in blocks:

        cleaned_steps = []

        for step in block:

            cleaned_step = []

            for solvent in step:

                cleaned = clean_solvent_component(solvent)

                if pd.isna(cleaned):
                    cleaned_step.append(pd.NA)
                else:
                    cleaned_step.append(cleaned)

            cleaned_steps.append(cleaned_step)

        cleaned_blocks.append(cleaned_steps)

    return cleaned_blocks


copy_df["Perovskite_deposition_solvents__parsed_clean"] = (
    copy_df["Perovskite_deposition_solvents__parsed"]
    .apply(clean_parsed_solvents)
)


# ============================================================
# 4. INSPECT @ CASES
# ============================================================

at_rows = copy_df[
    copy_df[solvent_col]
    .astype("string")
    .str.contains("@", regex=False, na=False)
][
    [
        solvent_col,
        "Perovskite_deposition_solvents_mixing_ratios",
        "Perovskite_deposition_solvents__structure_input",
        "Perovskite_deposition_solvents__parsed_clean"
    ]
].copy()

# Temporarily convert the list column to string to avoid the unhashable list TypeError
at_rows["Perovskite_deposition_solvents__parsed_clean_str"] = at_rows["Perovskite_deposition_solvents__parsed_clean"].astype(str)
at_rows = at_rows.drop_duplicates(subset=[
    solvent_col,
    "Perovskite_deposition_solvents_mixing_ratios",
    "Perovskite_deposition_solvents__structure_input",
    "Perovskite_deposition_solvents__parsed_clean_str"
]).drop(columns=["Perovskite_deposition_solvents__parsed_clean_str"])


print("=" * 100)
print("@ CASES")
print("=" * 100)

print(
    at_rows.to_string(index=False)
)


# ============================================================
# 5. INSPECT EMBEDDED MISSING VALUES
# ============================================================

missing_token_rows = copy_df[
    copy_df[solvent_col]
    .astype("string")
    .str.contains(
        r"(?i)(^|[;|>])\s*(none|unknown)\s*($|[;|>])",
        regex=True,
        na=False
    )
][
    [
        solvent_col,
        "Perovskite_deposition_solvents__parsed_clean",
        "Perovskite_deposition_solvents__flags"
    ]
]


print("\n" + "=" * 100)
print("EMBEDDED NONE / UNKNOWN CASES")
print("=" * 100)

print(
    missing_token_rows.to_string(index=False)
)

SOLVENT VALUES THAT CHANGED
Perovskite_deposition_solvents_original Perovskite_deposition_solvents_clean
                      Dimethylacetamide                                 DMAc
                  N,N-dimethylacetamide                                 DMAc
                      Dimethylacetamide                                 DMAc
                  N,N-dimethylacetamide                                 DMAc
                      Dimethylacetamide                                 DMAc
                      Dimethylacetamide                                 DMAc
                      Dimethylacetamide                                 DMAc
                      Dimethylacetamide                                 DMAc
                  N,N-dimethylacetamide                                 DMAc
                  N,N-dimethylacetamide                                 DMAc
                      Dimethylacetamide                                 DMAc
                      Dimethylacetamide         

/tmp/ipykernel_112139/4075182118.py:469: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


              Perovskite_deposition_solvents            Perovskite_deposition_solvents__parsed_clean Perovskite_deposition_solvents__flags
                                        none                                              [[[<NA>]]]                                      
                                        none                                              [[[<NA>]]]                                      
                                 DMF >> none                                       [[[DMF], [<NA>]]]                        multiple_steps
                                        none                                              [[[<NA>]]]                                      
                                        none                                              [[[<NA>]]]                                      
                                        none                                              [[[<NA>]]]                                      
                           

***`Perovskite_deposition_aggregation_state_of_reactants`*** parsing

In [10]:
import pandas as pd

VALID_STATES = {"Liquid", "Gas", "Solid", "Unknown"}


def parse_aggregation_state(value):
    if pd.isna(value):
        return pd.NA, {"missing"}

    raw = str(value).strip()

    if not raw:
        return pd.NA, {"empty_value"}

    flags = set()
    blocks = []

    # "|" separates blocks
    for block in raw.split("|"):
        block = block.strip()

        if not block:
            flags.add("empty_block")
            blocks.append([])
            continue

        states = []

        # ">>" represents sequential states
        for state in block.split(">>"):
            state = state.strip()

            if not state:
                flags.add("empty_state")
                continue

            if state not in VALID_STATES:
                flags.add("unexpected_state")

            states.append(state)

        blocks.append(states)

    if len(blocks) > 1:
        flags.add("multiple_blocks")

    if any(len(block) > 1 for block in blocks):
        flags.add("sequential_states")

    # Detect malformed delimiters
    if raw.startswith("|") or raw.endswith("|"):
        flags.add("leading_or_trailing_block_separator")

    if raw.startswith(">>") or raw.endswith(">>"):
        flags.add("leading_or_trailing_state_separator")

    if "||" in raw:
        flags.add("empty_block")

    if ">>>>" in raw:
        flags.add("empty_state")

    return blocks, flags


parsed = copy_df[
    "Perovskite_deposition_aggregation_state_of_reactants"
].apply(parse_aggregation_state)

copy_df[
    "Perovskite_deposition_aggregation_state_of_reactants__parsed"
] = parsed.apply(lambda x: x[0])

copy_df[
    "Perovskite_deposition_aggregation_state_of_reactants__flags"
] = parsed.apply(lambda x: ";".join(sorted(x[1])))

flagged = copy_df[
    copy_df[
        "Perovskite_deposition_aggregation_state_of_reactants__flags"
    ].astype("string").str.contains(
        "unexpected_state|empty_|leading_or_trailing",
        na=False
    )
]

print(
    flagged[
        [
            "Perovskite_deposition_aggregation_state_of_reactants",
            "Perovskite_deposition_aggregation_state_of_reactants__flags"
        ]
    ].to_string(index=False)
)

copy_df["Perovskite_deposition_aggregation_state_of_reactants"] = (
    copy_df["Perovskite_deposition_aggregation_state_of_reactants"]
    .replace("Unknown", pd.NA)
)

Empty DataFrame
Columns: [Perovskite_deposition_aggregation_state_of_reactants, Perovskite_deposition_aggregation_state_of_reactants__flags]
Index: []


***`Perovskite_deposition_procedure`*** parsing

In [11]:
import re
import pandas as pd


# ---------------------------------------------------------------------
# Perovskite deposition procedure terminology corrections
# ---------------------------------------------------------------------

DEPOSITION_REPLACEMENTS = {
    "Dipcoating": "Dipp-coating",
    "Drop-casting": "Dropcasting",
    "Spray coating": "Spray-coating",
    "Close space sublimation": "Closed space sublimation",
    "spin-coatng": "Spin-coating",
    "Springkling": "Sprinkling",
    "Inverse temperture crysilization": "Inverse temperature crystallization",
}


# ---------------------------------------------------------------------
# Clean individual procedure terms
# ---------------------------------------------------------------------

def _clean_deposition_term(term):
    term = term.strip()

    if term in DEPOSITION_REPLACEMENTS:
        return DEPOSITION_REPLACEMENTS[term]

    return term


# ---------------------------------------------------------------------
# Parse deposition procedure structure
# ---------------------------------------------------------------------

def parse_deposition_structure(value):

    # Missing value
    if pd.isna(value):
        return [], {"missing"}

    raw = str(value).strip()

    # Empty value
    if not raw:
        return [], {"empty_value"}

    flags = set()
    layers = []

    # Split separate blocks using |
    for layer in raw.split("|"):

        layer = layer.strip()

        if not layer:
            flags.add("empty_layer")
            layers.append([])
            continue

        procedures = []

        # Split sequential procedures using >>
        for procedure in layer.split(">>"):

            procedure = procedure.strip()

            if not procedure:
                flags.add("empty_procedure")
                continue

            cleaned = _clean_deposition_term(procedure)

            if cleaned != procedure:
                flags.add("terminology_corrected")

            procedures.append(cleaned)

        layers.append(procedures)

    # Structural flags
    if len(layers) > 1:
        flags.add("multi_layer")

    if any(len(layer) > 1 for layer in layers):
        flags.add("sequential_procedure")

    # Malformed delimiter checks
    if re.search(r"^\s*\|", raw) or re.search(r"\|\s*$", raw):
        flags.add("leading_or_trailing_layer_separator")

    if re.search(r"^\s*>>", raw) or re.search(r">>\s*$", raw):
        flags.add("leading_or_trailing_procedure_separator")

    if re.search(r"\|\s*\|", raw):
        flags.add("empty_layer")

    if re.search(r">>\s*>>", raw):
        flags.add("empty_procedure")

    return layers, flags


# ---------------------------------------------------------------------
# Reconstruct cleaned value
# ---------------------------------------------------------------------

def reconstruct_deposition(layers):

    if not layers:
        return pd.NA

    cleaned_layers = []

    for layer in layers:
        cleaned_layers.append(" >> ".join(layer))

    return " | ".join(cleaned_layers)


# ---------------------------------------------------------------------
# Apply parser
# ---------------------------------------------------------------------

column = "Perovskite_deposition_procedure"

parsed = copy_df[column].apply(
    parse_deposition_structure
)


# Parsed structure
copy_df[f"{column}__parsed"] = parsed.apply(
    lambda x: x[0]
)


# Flags
copy_df[f"{column}__flags"] = parsed.apply(
    lambda x: ";".join(sorted(x[1]))
)


# Cleaned / reconstructed value
copy_df[f"{column}_clean"] = (
    copy_df[f"{column}__parsed"]
    .apply(reconstruct_deposition)
)


# ---------------------------------------------------------------------
# Show only rows where the value changed
# ---------------------------------------------------------------------

changed = copy_df[
    copy_df[column].astype("string")
    != copy_df[f"{column}_clean"].astype("string")
]

print("=" * 100)
print("Perovskite_deposition_procedure")
print("=" * 100)

print(
    changed[
        [
            column,
            f"{column}_clean",
            f"{column}__flags"
        ]
    ].to_string(index=False)
)

Perovskite_deposition_procedure
                        Perovskite_deposition_procedure                    Perovskite_deposition_procedure_clean     Perovskite_deposition_procedure__flags
Spin-coating >> Spin-coating >> Close space sublimation Spin-coating >> Spin-coating >> Closed space sublimation sequential_procedure;terminology_corrected
Spin-coating >> Spin-coating >> Close space sublimation Spin-coating >> Spin-coating >> Closed space sublimation sequential_procedure;terminology_corrected
Spin-coating >> Spin-coating >> Close space sublimation Spin-coating >> Spin-coating >> Closed space sublimation sequential_procedure;terminology_corrected
Spin-coating >> Spin-coating >> Close space sublimation Spin-coating >> Spin-coating >> Closed space sublimation sequential_procedure;terminology_corrected
Spin-coating >> Spin-coating >> Close space sublimation Spin-coating >> Spin-coating >> Closed space sublimation sequential_procedure;terminology_corrected
Spin-coating >> Spin-coating >> 

***`Perovskite_deposition_quenching_media`*** parsing

In [12]:
def parse_quenching_media(value):
    if pd.isna(value):
        return pd.NA, {"missing"}

    raw = str(value).strip()

    if not raw:
        return pd.NA, {"empty_value"}

    flags = set()

    # First check capitalization-only corrections
    capitalization_replacements = {
        "chlorobenzene": "Chlorobenzene",
        "diethyl ether": "Diethyl ether",
        "ethyl acetate": "Ethyl acetate",
        "toluene": "Toluene",
        "anisole": "Anisole",
        "ethyl ether": "Ethyl ether",
        "ether": "Ether",
        "methyl acetate": "Methyl acetate",
        "isopropanol": "Isopropanol",
        "dichlorobenzene": "Dichlorobenzene",
        "acetonitrile": "Acetonitrile",
        "methanol": "Methanol",
        "isopropyl acetate": "Isopropyl acetate",
        "propyl acetate": "Propyl acetate",
        "acetic acid": "Acetic acid",
        "anhydrous chlorobenzene": "Anhydrous chlorobenzene",
        "anhydrous ethyl acetate": "Anhydrous ethyl acetate",
        "antisolvent": "Antisolvent",
    }

    # Actual spelling corrections
    spelling_replacements = {
        "diethylether": "Diethyl ether",
        "Triochloromethane": "Trichloromethane",
        "Di-n-butyl ehter": "Di-n-butyl ether",
        "Diclorobenzene": "Dichlorobenzene",
        "ethylacetate": "Ethyl acetate",
        "Flash infrared annealling": "Flash infrared annealing",
    }

    if raw in capitalization_replacements:
        cleaned = capitalization_replacements[raw]
        flags.add("capitalization_corrected")

    elif raw in spelling_replacements:
        cleaned = spelling_replacements[raw]
        flags.add("spelling_corrected")

    else:
        cleaned = raw

    return cleaned, flags

parsed = copy_df["Perovskite_deposition_quenching_media"].apply(
    parse_quenching_media
)

copy_df["Perovskite_deposition_quenching_media__parsed"] = (
    parsed.apply(lambda x: x[0])
)

copy_df["Perovskite_deposition_quenching_media__flags"] = (
    parsed.apply(lambda x: ";".join(sorted(x[1])))
)

changed = copy_df[
    copy_df["Perovskite_deposition_quenching_media"].astype("string")
    != copy_df["Perovskite_deposition_quenching_media__parsed"].astype("string")
]

print(
    changed[
        [
            "Perovskite_deposition_quenching_media",
            "Perovskite_deposition_quenching_media__parsed",
            "Perovskite_deposition_quenching_media__flags",
        ]
    ].to_string(index=False)
)

Perovskite_deposition_quenching_media Perovskite_deposition_quenching_media__parsed Perovskite_deposition_quenching_media__flags
                     Di-n-butyl ehter                              Di-n-butyl ether                           spelling_corrected
                    Triochloromethane                              Trichloromethane                           spelling_corrected
                    Triochloromethane                              Trichloromethane                           spelling_corrected
                        chlorobenzene                                 Chlorobenzene                     capitalization_corrected
                    Triochloromethane                              Trichloromethane                           spelling_corrected
                    Triochloromethane                              Trichloromethane                           spelling_corrected
                    Triochloromethane                              Trichloromethane              

"ETL_stack_sequence",
    "HTL_stack_sequence",
    "Backcontact_stack_sequence",
    "Cell_stack_sequence"

In [13]:
import re
import pandas as pd
import numpy as np


# ============================================================
# 1. SAFE MISSING-VALUE CHECK
# ============================================================

def is_missing_scalar(value):

    if value is None:
        return True

    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        return False

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# ============================================================
# 2. GENERAL RAW-VALUE CLEANING
# ============================================================

def clean_raw_value(value):

    # Already parsed structure
    if isinstance(value, (list, tuple, np.ndarray)):
        return value

    if is_missing_scalar(value):
        return None

    value = str(value)

    # Whitespace normalization only
    value = re.sub(r"\s+", " ", value).strip()

    if not value:
        return None

    return value


# ============================================================
# 3. PARSE STACK SEQUENCE
# ============================================================

def parse_stack_sequence(value):

    value = clean_raw_value(value)

    # Actual missing value
    if value is None:
        return [], {"missing"}

    # Already parsed
    if isinstance(value, (list, tuple, np.ndarray)):
        return value, {"already_parsed"}

    # Entire cell represents missing data
    if value.lower() in {
        "unknown",
        "none",
        "nan",
        "na",
        "n/a"
    }:
        return [], {"missing"}

    flags = set()

    # Normalize whitespace around layer separator
    raw = re.sub(r"\s*\|\s*", "|", value)

    # | = separate stack layers
    layers = raw.split("|")

    parsed_layers = []

    for layer in layers:

        layer = layer.strip()

        # Preserve empty layers as anomalies
        if not layer:
            flags.add("empty_layer")
            parsed_layers.append([])
            continue

        materials = []

        # ; = multiple materials within the same layer
        for material in re.split(r"\s*;\s*", layer):

            material = material.strip()

            if not material:
                flags.add("empty_material")
                continue

            # Missing material occurring inside a stack
            if material.lower() in {
                "unknown",
                "none",
                "nan",
                "na",
                "n/a"
            }:
                materials.append(pd.NA)
                flags.add("embedded_missing")
                continue

            # Preserve material name exactly
            materials.append(material)

        parsed_layers.append(materials)

    # ========================================================
    # STRUCTURAL FLAGS
    # ========================================================

    if len(parsed_layers) > 1:
        flags.add("multiple_layers")

    if any(len(layer) > 1 for layer in parsed_layers):
        flags.add("multiple_materials")

    if raw.startswith("|") or raw.endswith("|"):
        flags.add("leading_or_trailing_layer_separator")

    if "||" in raw:
        flags.add("empty_layer")

    if raw.startswith(";") or raw.endswith(";"):
        flags.add("leading_or_trailing_material_separator")

    if ";;" in raw:
        flags.add("empty_material")

    return parsed_layers, flags


# ============================================================
# 4. STACK COLUMNS
# ============================================================

stack_columns = [
    "ETL_stack_sequence",
    "HTL_stack_sequence",
    "Backcontact_stack_sequence",
    "Cell_stack_sequence"
]


# ============================================================
# 5. PRESERVE ORIGINAL VALUES
# ============================================================

for col in stack_columns:

    copy_df[f"{col}__original"] = copy_df[col].copy()


# ============================================================
# 6. PARSE WITHOUT OVERWRITING ORIGINAL DATA
# ============================================================

for col in stack_columns:

    parsed = copy_df[f"{col}__original"].apply(
        parse_stack_sequence
    )

    copy_df[f"{col}__parsed"] = parsed.apply(
        lambda x: x[0]
    )

    copy_df[f"{col}__flags"] = parsed.apply(
        lambda x: ";".join(sorted(x[1]))
    )


# ============================================================
# 7. FINAL FLAG SUMMARY
# ============================================================

for col in stack_columns:

    print("\n")
    print("=" * 100)
    print(f"{col} — FINAL FLAG SUMMARY")
    print("=" * 100)

    flags = (
        copy_df[f"{col}__flags"]
        .replace("", pd.NA)
        .dropna()
        .str.split(";")
        .explode()
        .value_counts()
    )

    print(flags)


# ============================================================
# 8. CHECK FOR ACTUAL STRUCTURAL PROBLEMS
# ============================================================

problem_flags = {
    "empty_layer",
    "empty_material",
    "leading_or_trailing_layer_separator",
    "leading_or_trailing_material_separator",
    "embedded_missing"
}


for col in stack_columns:

    flagged = copy_df[
        copy_df[f"{col}__flags"]
        .apply(
            lambda x: any(
                flag in problem_flags
                for flag in str(x).split(";")
            )
        )
    ][
        [
            f"{col}__original",
            f"{col}__parsed",
            f"{col}__flags"
        ]
    ]

    print("\n")
    print("=" * 100)
    print(f"{col} — ACTUAL STRUCTURAL PROBLEMS")
    print("=" * 100)

    print(
        flagged.to_string(index=False)
    )



ETL_stack_sequence — FINAL FLAG SUMMARY
ETL_stack_sequence__flags
multiple_layers       27564
missing                 765
multiple_materials      612
Name: count, dtype: int64


HTL_stack_sequence — FINAL FLAG SUMMARY
HTL_stack_sequence__flags
missing               3526
multiple_layers       2445
multiple_materials     404
Name: count, dtype: int64


Backcontact_stack_sequence — FINAL FLAG SUMMARY
Backcontact_stack_sequence__flags
multiple_layers       3251
missing                815
multiple_materials     115
Name: count, dtype: int64


Cell_stack_sequence — FINAL FLAG SUMMARY
Cell_stack_sequence__flags
multiple_layers       48433
multiple_materials     1099
missing                 570
embedded_missing          2
Name: count, dtype: int64


ETL_stack_sequence — ACTUAL STRUCTURAL PROBLEMS
Empty DataFrame
Columns: [ETL_stack_sequence__original, ETL_stack_sequence__parsed, ETL_stack_sequence__flags]
Index: []


HTL_stack_sequence — ACTUAL STRUCTURAL PROBLEMS
Empty DataFrame
Columns: [H

***`Substrate_stack_sequence`*** parsing

In [14]:
import re
import pandas as pd


# ============================================================
# SUBSTRATE STACK SEQUENCE — CLEANING
# ============================================================

column = "Substrate_stack_sequence"


# ============================================================
# 1. CLEANING FUNCTION
# ============================================================

def clean_substrate_stack(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if not value:
        return pd.NA

    # --------------------------------------------------------
    # Correct capitalization of "glass"
    # --------------------------------------------------------
    value = re.sub(
        r"\bglass\b",
        "Glass",
        value
    )

    # --------------------------------------------------------
    # Convert "/" into the official layer separator
    # --------------------------------------------------------
    value = re.sub(
        r"\s*/\s*",
        " | ",
        value
    )

    # --------------------------------------------------------
    # Normalize spacing around "|"
    # --------------------------------------------------------
    value = re.sub(
        r"\s*\|\s*",
        " | ",
        value
    )

    return value


# ============================================================
# 2. CREATE CLEAN VERSION
# ============================================================

copy_df[f"{column}_clean"] = (
    copy_df[column]
    .apply(clean_substrate_stack)
)


# ============================================================
# 3. PREVIEW CHANGES
# ============================================================

changed = copy_df[
    copy_df[column].astype("string")
    != copy_df[f"{column}_clean"].astype("string")
]

print("=" * 100)
print(f"{column} — CLEANING PREVIEW")
print("=" * 100)

print("Changed rows:", len(changed))

print(
    changed[
        [
            column,
            f"{column}_clean"
        ]
    ]
    .drop_duplicates()
    .to_string(index=False)
)


# ============================================================
# 4. APPLY CLEANED VALUES
# ============================================================

copy_df[column] = copy_df[f"{column}_clean"]


# ============================================================
# 5. VERIFY
# ============================================================

print("\n" + "=" * 100)
print(f"{column} — CLEANING COMPLETE")
print("=" * 100)

print("Original rows:", len(copy_df))
print("Missing values:", copy_df[column].isna().sum())
print("Unique non-missing values:", copy_df[column].nunique(dropna=True))

Substrate_stack_sequence — CLEANING PREVIEW
Changed rows: 134
       Substrate_stack_sequence  Substrate_stack_sequence_clean
             Willow glass | ITO              Willow Glass | ITO
             Willow glass | IZO              Willow Glass | IZO
             Willow glass | AZO              Willow Glass | AZO
Willow glass | Graphene | Ag-nw Willow Glass | Graphene | Ag-nw
                        PEN/ITO                       PEN | ITO
                        PET/ITO                       PET | ITO
                        ITO/PET                       ITO | PET
                        ITO/PEN                       ITO | PEN
          Ti metal-coated glass           Ti metal-coated Glass
                      ITO/Glass                     ITO | Glass
                        SiO2/Si                       SiO2 | Si
                        PET/IMI                       PET | IMI
                        LiF/FTO                       LiF | FTO
                      TCO/glass           

In [15]:
import re
import pandas as pd


# ---------------------------------------------------------------------
# ETL / HTL deposition procedure terminology corrections
# ---------------------------------------------------------------------

DEPOSITION_REPLACEMENTS = {
    "Spray-pyrolys": "Spray-pyrolysis",
    "Screen printnig": "Screen printing",
    "Dipp ccoating": "Dipp-coating",
    "Hydorthermal": "Hydrothermal",
    "Oxdation": "Oxidation",
    "Electrohemical anodization": "Electrochemical anodization",

    # Evaporation terminology
    "E-beam evaporation": "Electron-beam evaporation",
    "Electron beam evaporation": "Electron-beam evaporation",
}


# ---------------------------------------------------------------------
# Clean individual procedure terms
# ---------------------------------------------------------------------

def _clean_deposition_term(term):
    """
    Apply only explicitly defined terminology corrections.

    No synonym merging or interpretation of rare procedures.
    """
    term = term.strip()

    if term in DEPOSITION_REPLACEMENTS:
        return DEPOSITION_REPLACEMENTS[term]

    return term


# ---------------------------------------------------------------------
# Parse deposition procedure structure
# ---------------------------------------------------------------------

def parse_deposition_structure(value):
    """
    Parse ETL / HTL deposition procedure.

    Structure:
        |  = separate procedure blocks
        >> = sequential procedures within a block
    """

    # Missing value
    if pd.isna(value):
        return [], {"missing"}

    raw = str(value).strip()

    # Empty value
    if not raw:
        return [], {"empty_value"}

    flags = set()
    layers = []

    # -------------------------------------------------------------
    # 1. Split into blocks using |
    # -------------------------------------------------------------

    for layer in raw.split("|"):

        layer = layer.strip()

        if not layer:
            flags.add("empty_layer")
            layers.append([])
            continue

        procedures = []

        # ---------------------------------------------------------
        # 2. Split sequential procedures using >>
        # ---------------------------------------------------------

        for procedure in layer.split(">>"):

            procedure = procedure.strip()

            if not procedure:
                flags.add("empty_procedure")
                continue

            cleaned = _clean_deposition_term(procedure)

            if cleaned != procedure:
                flags.add("terminology_corrected")

            procedures.append(cleaned)

        layers.append(procedures)

    # -------------------------------------------------------------
    # 3. Structural flags
    # -------------------------------------------------------------

    if len(layers) > 1:
        flags.add("multi_layer")

    if any(len(layer) > 1 for layer in layers):
        flags.add("sequential_procedure")

    # -------------------------------------------------------------
    # 4. Malformed delimiter checks
    # -------------------------------------------------------------

    if re.search(r"^\s*\|", raw) or re.search(r"\|\s*$", raw):
        flags.add("leading_or_trailing_layer_separator")

    if re.search(r"^\s*>>", raw) or re.search(r">>\s*$", raw):
        flags.add("leading_or_trailing_procedure_separator")

    if re.search(r"\|\s*\|", raw):
        flags.add("empty_layer")

    if re.search(r">>\s*>>", raw):
        flags.add("empty_procedure")

    return layers, flags


# ---------------------------------------------------------------------
# Reconstruct cleaned value
# ---------------------------------------------------------------------

def reconstruct_deposition(layers):

    if not layers:
        return pd.NA

    cleaned_layers = []

    for layer in layers:
        cleaned_layers.append(" >> ".join(layer))

    return " | ".join(cleaned_layers)


# ---------------------------------------------------------------------
# Apply parser to ETL and HTL
# ---------------------------------------------------------------------

procedure_columns = [
    "ETL_deposition_procedure",
    "HTL_deposition_procedure",
]


for column in procedure_columns:

    parsed = copy_df[column].apply(
        parse_deposition_structure
    )

    # Parsed nested structure
    copy_df[f"{column}__parsed"] = parsed.apply(
        lambda x: x[0]
    )

    # Flags
    copy_df[f"{column}__flags"] = parsed.apply(
        lambda x: ";".join(sorted(x[1]))
    )

    # Reconstructed cleaned value
    copy_df[f"{column}_clean"] = (
        copy_df[f"{column}__parsed"]
        .apply(reconstruct_deposition)
    )


# ---------------------------------------------------------------------
# Show only rows where the value changed
# ---------------------------------------------------------------------

for column in procedure_columns:

    clean_column = f"{column}_clean"
    flags_column = f"{column}__flags"

    changed = copy_df[
        copy_df[column].astype("string")
        != copy_df[clean_column].astype("string")
    ]

    print("\n" + "=" * 100)
    print(column)
    print("=" * 100)

    print(
        changed[
            [
                column,
                clean_column,
                flags_column,
            ]
        ].to_string(index=False)
    )


ETL_deposition_procedure
                                             ETL_deposition_procedure                                          ETL_deposition_procedure_clean                        ETL_deposition_procedure__flags
                                         Spray-pyrolys | Spin-coating                                          Spray-pyrolysis | Spin-coating                      multi_layer;terminology_corrected
                                                        Spray-pyrolys                                                         Spray-pyrolysis                                  terminology_corrected
                                         Spray-pyrolys | Spin-coating                                          Spray-pyrolysis | Spin-coating                      multi_layer;terminology_corrected
                                         Spray-pyrolys | Spin-coating                                          Spray-pyrolysis | Spin-coating                      multi_layer;terminology

In [16]:
import re
import pandas as pd

# ============================================================
# BACKCONTACT DEPOSITION PROCEDURE — APPROVED CORRECTIONS
# ============================================================

DEPOSITION_REPLACEMENTS = {
    # Spray coating
    "Spray coating": "Spray-coating",

    # Drop casting
    "Dropp casting": "Dropcasting",
    "Drop-casting": "Dropcasting",

    # Candle burning
    "Candel burning": "Candle burning",

    # Sprinkling
    "Springkling": "Sprinkling",

    # Evaporation terminology
    "E-beam evaporation": "Electron-beam evaporation",
    "Electron beam evaporation": "Electron-beam evaporation",
}


def _clean_deposition_term(term):
    term = term.strip()

    if term in DEPOSITION_REPLACEMENTS:
        return DEPOSITION_REPLACEMENTS[term]

    return term


def parse_deposition_structure(value):

    if pd.isna(value):
        return [], {"missing"}

    raw = str(value).strip()

    if not raw:
        return [], {"empty_value"}

    flags = set()
    layers = []

    for layer in raw.split("|"):

        layer = layer.strip()

        if not layer:
            flags.add("empty_layer")
            layers.append([])
            continue

        procedures = []

        for procedure in layer.split(">>"):

            procedure = procedure.strip()

            if not procedure:
                flags.add("empty_procedure")
                continue

            cleaned = _clean_deposition_term(procedure)

            if cleaned != procedure:
                flags.add("terminology_corrected")

            procedures.append(cleaned)

        layers.append(procedures)

    # Structural information
    if len(layers) > 1:
        flags.add("multi_layer")

    if any(len(layer) > 1 for layer in layers):
        flags.add("sequential_procedure")

    if re.search(r"^\s*\|", raw) or re.search(r"\|\s*$", raw):
        flags.add("leading_or_trailing_layer_separator")

    if re.search(r"^\s*>>", raw) or re.search(r">>\s*$", raw):
        flags.add("leading_or_trailing_procedure_separator")

    if re.search(r"\|\s*\|", raw):
        flags.add("empty_layer")

    if re.search(r">\>\s*>>", raw):
        flags.add("empty_procedure")

    return layers, flags


def reconstruct_deposition(layers):

    if not layers:
        return pd.NA

    cleaned_layers = []

    for layer in layers:
        cleaned_layers.append(" >> ".join(layer))

    return " | ".join(cleaned_layers)


# ============================================================
# PRESERVE ORIGINAL
# ============================================================

column = "Backcontact_deposition_procedure"

copy_df[f"{column}__original"] = copy_df[column].copy()


# ============================================================
# PARSE
# ============================================================

parsed = copy_df[f"{column}__original"].apply(
    parse_deposition_structure
)

copy_df[f"{column}__parsed"] = parsed.apply(
    lambda x: x[0]
)

copy_df[f"{column}__flags"] = parsed.apply(
    lambda x: ";".join(sorted(x[1]))
)


# ============================================================
# RECONSTRUCT CLEAN VERSION
# ============================================================

copy_df[f"{column}_clean"] = (
    copy_df[f"{column}__parsed"]
    .apply(reconstruct_deposition)
)


# ============================================================
# SHOW ONLY CHANGED ROWS
# ============================================================

changed = copy_df[
    copy_df[f"{column}__original"].astype("string")
    != copy_df[f"{column}_clean"].astype("string")
]

print("=" * 100)
print(column)
print("=" * 100)

print("Changed rows:", len(changed))

print(
    changed[
        [
            f"{column}__original",
            f"{column}_clean",
            f"{column}__flags"
        ]
    ].to_string(index=False)
)

Backcontact_deposition_procedure
Changed rows: 143
                                                     Backcontact_deposition_procedure__original                                                                                      Backcontact_deposition_procedure_clean                Backcontact_deposition_procedure__flags
                                                                   Candel burning | Sandwiching                                                                                                Candle burning | Sandwiching                      multi_layer;terminology_corrected
                                                                   Candel burning | Sandwiching                                                                                                Candle burning | Sandwiching                      multi_layer;terminology_corrected
                                                                   Candel burning | Sandwiching                             

In [17]:
copy_df["Cell_architecture"] = (
    copy_df["Cell_architecture"]
    .replace("Unknown", pd.NA)
)
copy_df["Cell_architecture"] = (
    copy_df["Cell_architecture"]
    .astype("string")
)
copy_df["Cell_architecture"].unique()

<StringArray>
['nip', 'Schottky', 'pin', <NA>, 'Back contacted', 'Pn-Heterojunction', 'Front contacted', 'Other']
Length: 8, dtype: string

In [18]:
boolean_columns = [
    "Perovskite_composition_inorganic",
    "Perovskite_composition_leadfree",
    "Perovskite_composition_perovskite_inspired_structure",
    "Perovskite_composition_perovskite_ABC3_structure",
    "Perovskite_dimension_3D",
    "Perovskite_single_crystal",
    "Perovskite_band_gap_graded",
    "Perovskite_deposition_solvent_annealing",
    "Perovskite_deposition_quenching_induced_crystallisation",
    "Cell_flexible",
    "Cell_semitransparent"
]

In [19]:
for column in boolean_columns:
    copy_df[column] = copy_df[column].astype("boolean")

In [20]:
copy_df = copy_df.copy()

copy_df = copy_df.map(
    lambda x: x.strip() if isinstance(x, str) else x
)

copy_df = copy_df.replace(
    r"^\s*$",
    pd.NA,
    regex=True
)

copy_df = copy_df.replace(
    r"(?i)^nan$",
    pd.NA,
    regex=True
)

copy_df = copy_df.fillna(pd.NA)

Remove the duplicates
Remove the unknown pCE values
Column drop
Go for outlier Checks
Encoding

In [23]:
composition_cols = [
    "Perovskite_composition_short_form",
    "Perovskite_composition_long_form",
    "Perovskite_composition_a_ions",
    "Perovskite_composition_a_ions_coefficients",
    "Perovskite_composition_b_ions",
    "Perovskite_composition_b_ions_coefficients",
    "Perovskite_composition_c_ions",
    "Perovskite_composition_c_ions_coefficients",
]

architecture_cols = [
    "Cell_architecture",
    "Cell_stack_sequence",
    "Substrate_stack_sequence",
    "ETL_stack_sequence",
    "HTL_stack_sequence",
    "Backcontact_stack_sequence",
]

fabrication_cols = [
    "Perovskite_thickness",
    "Perovskite_deposition_number_of_deposition_steps",
    "Perovskite_deposition_aggregation_state_of_reactants",
    "Perovskite_deposition_procedure",
    "Perovskite_deposition_solvents",
    "Perovskite_deposition_solvents_mixing_ratios",
    "Perovskite_deposition_thermal_annealing_temperature",
    "Perovskite_deposition_thermal_annealing_time",
    "Perovskite_deposition_synthesis_atmosphere",
    "Perovskite_deposition_solvent_annealing",
    "Perovskite_deposition_quenching_induced_crystallisation",
    "Perovskite_deposition_quenching_media",
    "ETL_deposition_procedure",
    "ETL_thickness",
    "HTL_deposition_procedure",
    "HTL_thickness",
    "Backcontact_deposition_procedure",
    "Backcontact_thickness_list",
]

Number of columns used: 33
Rows in dataset: 49027
Rows belonging to duplicate groups: 32783
Exact duplicate rows after keeping first: 24015
Unique records based on candidate columns: 25012
